# Binarna klasifikacija kori??enjem neuronskih mre?a

Ova sveska uvodi Keras biblioteku i ilustruje njene osnovne koncepte. Zadatak na kojem ?emo raditi je kreiranje neuronske mre?e sa propagacijom unapred (engl. feed forward neural network) koja se mo?e iskoristiti u zadatku binarne klasifikacije pacijenata sa dijabetesom.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
np.random.seed(7)

## Keras biblioteka

[Keras](https://keras.io/) je popularna biblioteka koja nudi podr?ku u radu sa dubokim neuronskim mre?ama. Na njenom razvoju radi Google tim, a zajednica koja je koristi je mnogobrojna. Da bismo mogli da je koristimo u okviru Anaconda okru?enja, potrebno ju je instalirati komandom `conda install -c conda-forge keras` u skladu sa [zvani?nim smernicama](https://anaconda.org/conda-forge/keras). Kako je njen API visokog nivoa, u pozadini, na niskom nivou, se mo?e koristiti Tensor Flow, Theano ili CNTK (tzv. backend deo biblioteke). Cilj pobrojanih biblioteka je da pokriju zadatke poput efikasnih algebarskih operacija i numeri?og difrerenciranja ili paralelilazija jer rad sa neuronskim mre?ama obi?no podrazumeva i rad sa velikom koli?inom podataka. U na?em radu ?e to biti Tensor Flow pa je potrebno osigurati se da je i on instaliran. Vi?e o instalaciji Tensor Flow biblioteke mo?ete pro?itati [ovde](https://anaconda.org/conda-forge/tensorflow).   

Keras podr?ava dve vrste API-ja, `standardni` koji ?emo upoznati u primerima koji slede i `funkcionalni` koji daje ne?to vi?e fleksibilnosti za nadgradnje biblioteke. Njega ?emo upoznati u pri?i o konvolutivnim neuronskim mre?ama.

In [ ]:
from tensorflow import keras

Za sada ?emo u?itati funkcije `Sequential` i `Dense` ?ije ?emo zna?enje pojasniti u nastavku.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense

## Skup podataka - Pima Indians Diabetes

Skup podataka "Pima Indians Diabetes" se sastoji od 768 instanci. Svaka instanca sadr?i vrednosti 8 medicinski relevantnih vrednosti uz pridru?enu klasu sa vrednostima 1 ili 0 u zavisnosti od toga da li je pacijentu u poslednjih 5 godina dijagnostifikovan dijabetes ili ne. 

Svi atributi su numeri?kog tipa sa opsegom vrednosti koji varira od atributa do atributa. 

Skup podataka se mo?e preuzeti sa platforme Kaggle sa [ove](https://www.kaggle.com/uciml/pima-indians-diabetes-database#diabetes.csv) adrese.

In [ ]:
data = pd.read_csv('data/pima_indians_diabetes.csv')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.describe()

Na? cilj je da napravimo model koji ?e na osnovu pomenutih atributa vr?iti predikciju da li pacijentkinja boluje od dijabetesa ili ne.

## Kreiranje neuronske mre?e

### Korak 1: Priprema skupa podataka

Kolona `Outcome` ukazuje da li pacijentkinja ima dijabetes (vrednost 1) ili ne (vrednost 0) i nju ?emo koristiti kao ciljnu promenljivu. Preostale kolone predstavljaju razli?ite medicinske podatke i njih ?emo koristiti kao atribute nad kojima ?e model biti obu?avan.

In [ ]:
y = data['Outcome']
X = data.drop(columns=['Outcome'])

Podatke ?emo uobi?ajeno podeliti funkcijom `train_test_split` biblioteke `scikit-learn`. Prilikom podele skupa podataka vodi?emo ra?una o stratifikaciji. 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

Podatke ?emo zbog numeri?ke stabilnosti mre?e kao i do sada standardizovati.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

### Korak 2: Kreiranje mre?e

Jedan neuron neuronske mre?e agregira vrednosti svih signala koji sti?u do njega i proizvodi odgovaraju?i izlazni signal. Signale koje neuron obra?uje nazivamo ulazima neurona (na slici su ulazi obele?eni sa $x_i$). Kako nisu svi signali od iste va?nosti, njih u ovoj matemati?koj apstrakciji prate odgovaraju?e numeri?ke te?ine (na slici su te?ine obele?ene sa $w_i$). Svaki neuron karakteri?e i poseban slobodni ?lan (engl. bias term) kojim se uti?e na specifi?na pona?anja neurona (na slici je ovaj ?lan obele?en slovom $b$). Tako, neuron obra?uje te?insku sumu svojih ulaza (na slici obele?enu sa $\Sigma$) uve?anu za vrednost slobodnog ?lana. Posebno, slobodni ?lan se mo?e posmatrati i kao ulaz neurona kojem je pridru?ena te?ina 1 zbog jednostavnijih numeri?kih zapisa. Ovako dobijena vrednost se dalje propu?ta kroz aktivacionu funkciju (na slici obele?enu sa $\phi$) i dobija se izlaz nurona $y$.  

<img src="assets/neuron-math.png" width=550>

Mre?a se sastoji od pojedina?nih slojeva neurona. Mo?emo re?i da su slojevi moduli koji obra?uju podatke i da za zadate ulaze generi?u odgovaraju?e izlaze. Razlikujemo ulazni, prvi, sloj mre?e, izlazni, poslednji, sloj mre?e i slojeve izme?u koje nazivamo skrivenim slojevima. <img src='assets/layers.png'> Slojeve karakteri?e matrica koeficijenata ?ije optimalne vrednosti mre?a u toku treniranja treba da nau?i. Samu matricu koeficijenata zovemo `kernel` matricom. Njene vrednosti se mogu nasumi?no inicijalizovati, ali se u praksi obi?no koriste neke bolje tehnike inicijalizacije koje dovode do boljeg treniranja mre?e. Trenutno najpoznatije su Gloriot (ili Xaviar) i He, obe nazvane po imenima autora. 

Potpuno povezani sloj (engl. fully connected layer), kao ?to ime ka?e, povezuju svaki ?vor, neuron, ulaznog sloja sa svakim ?vorom, neuronom, izlaznog sloja. Funkcija koju izra?unava mo?e se opisati formulom `output = activation(dot(input, kernel) + bias)` gde  `kernel` predstavlja matricu sloja, `bias` vektor njegovih slobodnih ?lanova, `dot` matri?no mno?enje, a `activation` aktivacionu funkciju koja se koristi na nivo sloja. 
<img src='assets/fully_connected_layer_2.png' style='width: 300px'>

Mre?u ?emo napraviti mre?u tako da njena arhitektura zadovoljava slede?e specifikacije:

* **Sloj 1**: potpuno povezani sloj sa 15 neurona i *ReLu* aktivacionom funkcijom 
* **Sloj 2**: potpuno povezani sloj sa 8 neurona i *ReLu* aktivacionom funkcijom
* **Sloj 3**: potpuno povezani sloj sa jednim neuronom i *sigmoidnom* aktivacionom funkcijom


Neuronska mre?a sa propagacijom unapred kreira se funkcijom `Sequential`. Na ovaj na?in nagla?avamo da ?emo slojeve mre?e dodavati jedan za drugim, povezivaju?i izlaze prethodnog sloja sa ulazima narednog sloja.

In [ ]:
model = Sequential()

Ulazni, prvi sloj mre?e, mo?emo kreirati pozivom funkcije `InputLayer`. Njen argument `input_shape` ukazuje na dimenziju ulaza, pa ?emo stoga postaviti da je njegova vrednost `(8, )` jer skup sa kojim radimo ima 8 atributa. Druga dimenzija para je fleksibilna i prilikom treniranja mre?e prilago?ava?e se veli?ini paketi?a koji se koristi. Na nivou Keras biblioteke, pojedina?ni slojevi mre?e dodaju se funkcijom `add`.

In [ ]:
model.add(InputLayer(input_shape=(8,)))

Potpuno povezani sloj (koji se naziva i gusti sloj) kreiramo funkcijom `Dense`. Kroz parametar `units`, ?ije se ime ?esto i izostavlja jer se novodi prvi u pozivu funkcije, navodi se broj izlaznih neurona. Biblioteka podrazumevano izlaze prethodnih slojeva uparuje sa ulazima narednih slojeva pa nije neophodno navoditi dimenziju ulaza. Za to se ina?e mo?e koristiti atribut `input_dim`. Ime aktivacione funkcije se navodi parametrom `activation` i to mogu biti sigmoidna funkcija, hiperboli?ki tanges, linearna funkcija, ReLu, softmax i jo? neke druge. Ostalim parametrima funkcije `Dense` mo?e se uticati na na?in inicijalizovanja kernel matrice i vektora slobodnih ?lanova koji se podrazumevano dodaje, kao i na njihove regularizacije.

Prate?i specifikaciju arhitekture, mo?emo dodati zahtevane slojeve. 

In [ ]:
model.add(Dense(units=15, activation='relu'))
model.add(Dense(units=8, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))

Opis mre?e po slojevima se mo?e dobiti funkcijom `summary`. Ona nam daje uvid i u ukupan broj parametara mre?e koje treba nau?iti. 

In [ ]:
model.summary()

Prokomentari?imo kratko da dobijene vrednosti odra?avaju prisustvo slobodnog ?lana u svakom sloju mre?e kada je u ulazi ulaznog sloja (135 = 8x15 + 1x15, 128 = 15x8 + 1x8, 9 = 8x1 + 1x1).  

Korak zadavanja ulaznog sloja funkcijom `InputLayer` i korak zadavanja prvog gustog sloja funkcijom `Dense` je u praksi mogu?e objediniti dodavanjem parametar `input_dim` prvom gustom sloju kojim se zadaje dimenzija ulaza. Tako se u prethodnom bloku koda umesto linija 

`model.add(InputLayer(input_shape=(8,)))
model.add(Dense(units=15, activation='relu'))
`

mogla navesti linija `model.add(Dense(units=15, input_dim=8, activation='relu'))`. 
Prvi na?in je ipak ?itljiviji i predstavlja novinu koja je uvedena sa novijom verzijom Keras biblioteke.


### Korak 3: Treniranje mre?e

Da bismo mogli da traniramo mre?u, neophodno je da se opredelimo za funkciju gubitka koja pristaje zadatku i odgovaraju?i optimizator. Funkcija gubitka (engl. loss function) izra?unava koliko dobro mre?a predvi?a ono ?to se od nje o?ekuje i generi?e povratni signal neophodan za samo u?enje. Optimizator dalje na osnovu ovako dobijenih vrednosti a?urira vrednosti parametara mre?e. Cilj treniranja mre?e je minimizovati funkciju gubitka uz o?uvanje dobre generalizacije. 
<img src='assets/training_steps.png'>

Mre?u ?emo trenirati prema slede?im smernicama:
    
* **Funkcija gre?ke**: binarna unakrsna entropija (engl. binary cross entropy)

* **Optimizator**: Adam

* **Broj epoha**: 50
* **Veli?ina paketi?a za treniranje (engl. batch size)**: 32

Funkcija `compile` se koristi na nivou modela za pridru?ivanje funkcije gubitka, optimizatora i, opciono, metrike koja ?e se koristiti za evaluaciju mre?e.

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

Prilikom treniranja, mre?i se prosle?uju takozvani `batch` paketi?i podataka. Nakon svakog obra?enog paketi?a, izra?unava se vrednost funkcije gubitka, a potom i a?uriraju parametri mre?e. Epoha predstavlja jedan prolazak kroz ceo skup podataka.  

Funkcijom `fit` se trenira sama mre?a. Ona o?ekuje podatke za treniranje, broj epoha (parametar `epochs`) i veli?inu paketi?a podataka (parametar `batch_size`). Objekat tipa `History` koji se dobija kao rezultat poziva ove funkcije sadr?i vrednosti funkcije gubitka i odabrane metrike na kraju svake epohe koje se, dalje, mogu koristiti za pra?enje procesa treniranja. Ove vrednosti se nalaze u re?niku sa imenom `history`.

In [ ]:
NUMBER_OF_EPOCHS = 50
BATCH_SIZE = 32

In [ ]:
history = model.fit(X_train, y_train, epochs=NUMBER_OF_EPOCHS, batch_size=BATCH_SIZE, verbose=0)

In [ ]:
plt.title('Treniranje mre?e - funkcija gubitka')
plt.xlabel('Epoha')
plt.ylabel('Funkcija gubitka')
plt.plot(history.epoch, history.history['loss'])
plt.show()

Na primer, na osnovu prvog grafika mogu se izvesti neke intuicije o optimizaciji, veli?ini koraka u?enja i brzini konvergencije. Slede?a slika sumira zaklju?ke i iskustva.  <img src='assets/loss_monitoring.png'>

In [ ]:
plt.title('Treniranje mre?e - ta?nost modela')
plt.xlabel('Epoha')
plt.ylabel('Ta?nost')
plt.plot(history.epoch, history.history['accuracy'])
plt.show()

U slede?em primeru, nakon uvo?enja validacionog skupa, diskutova?emo jo? neke analize.

### Korak 4: Evaluacija mre?e

Da bismo mogli da ocenimo uspe?nost treniranja mre?e, potrebne su nam vrednosti relevantnih metrika na skupu za testiranje. I u slu?aju neuronskih mre?a, ba? kao i do sada, prilikom evaluacije modela vodimo ra?una da ni u jednom trenutku ne do?e do preklapanja skupa za treniranje i skupa za testiranje. Evaluacija modela se radi propu?tanjem paketi?a skupa za testiranje kroz mre?u, bele?enjem i uprose?avanjem dobijenih vrednosti.

U na?em slu?aju, za zadatak klasifikacije, koristili smo kao metriku ta?nost.

In [ ]:
model.metrics_names

Funkcija `evaluate` biblioteke Keras nam omogu?ava da sprovedemo opisanu proceduru. Njen rezultat je niz koji sadr?i izra?unate vrednosti za svaku od metrika navedenih u `model.metrics_names` nizu.

In [ ]:
test_scores = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE)
print('Skup za testiranje {0}: {1}'.format(model.metrics_names[1], test_scores[1]))

Da bi se pratilo preprilago?avanje mre?e, obi?no se relevantne metrike ra?unaju i na skupu za treniranje.

In [ ]:
train_scores = model.evaluate(X_train,y_train)
print('Skup za treniranje {0}: {1}'.format(model.metrics_names[1], train_scores[1]))

#### Unakrsna validacija 

Nasumi?na podela skupa podataka na skup za treniranje i skup za testiranje i kod neuronskih mre?a, ba? kao i kod drugih modela, mo?e prouzrokovati neobjektivnost ocena. Unakrsnu validaciju je mogu?e realizovati podelom skupa podataka na grupe funkcijom `KFold` i pra?enjem dobijenih vrednosti. Ovde ?emo jednostavnosti radi upoznati omota? `KerasClassifier` koji omogu?ava unakrsnu validaciju Keras modela u duhu biblioteke `scikit-learn`.  

In [ ]:
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn import model_selection
from sklearn.pipeline import Pipeline
from sklearn import metrics

In [ ]:
def baseline_model():
    model = Sequential()
    model.add(Dense(units=15, input_dim=8, activation='relu'))
    model.add(Dense(units=8, activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])    
    return model

Funkcija KerasClassifier o?ekuje model koji treba pokrenuti, broj epoha i veli?inu paketi?a.

In [ ]:
classifier = KerasClassifier(build_fn=baseline_model, nb_epoch=50, batch_size=32, verbose=0)

In [ ]:
pipeline = Pipeline(steps=[('scaler', StandardScaler()), ('classifier', classifier)])

In [ ]:
accuracy_scores = model_selection.cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

In [ ]:
accuracy_scores

In [ ]:
np.average(accuracy_scores)

U radu sa velikim skupovima podataka, zbog vremenske zahtevnosti, ne uzimaju se velike vrednosti `cv` parametra, a vrlo ?esto se ovaj vid evaluacije i preska?e.

### Korak 5: Kori??enje mre?e

Za predikciju vrednosti modela mo?e se koristiti funkcija `predict`.

In [ ]:
number_of_features = X_train.shape[1]

In [ ]:
x_new = np.random.random(number_of_features)
x_new = scaler.transform(x_new.reshape(1, -1))

Vrednost koju na?a mre?a predvi?a je vrednost sigmoidne funkcije poslednjeg sloja pa da bi odredili da li nova instanca pripada klasi ili ne treba uporediti njenu vrednost sa 0.5. Podsetimo se da se ova vrednost mo?e interpretirati i kao verovatno?a pripadanja odre?enoj klasi.

In [ ]:
y_new_sigmoid = model.predict(x_new)

In [ ]:
y_new_sigmoid

In [ ]:
y_new = 0 if y_new_sigmoid<0.5 else 1

In [ ]:
y_new

Ovaj izraz se mo?e zapisati i na slede?i na?in:

In [ ]:
y_new = np.int(y_new_sigmoid<0.5)

In [ ]:
y_new

Jo? jedan, za nijansu kompaktniji, na?in koji omogu?ava o?itavanje predikcije je uz kori??enje funkcije `argmax`.

In [ ]:
np.argmax(model.predict(x_new), axis=-1)

U praksi se optimalna arhitektura mre?e tj. broj slojeva, broj neurona po slojevima, odgovaraju?e aktivacione funkcije, broj epoha treniranja i sli?no odre?uju kao i do sada pominjani hiperparametri. O tome ?e biti vi?e re?i u nastavku. 

<div class="alert alert-info">
Za ve?bu:

<ul>
    <li>Isprobati razli?ite aktivacione funkcije u slojevima, na primer, sigmoidnu i hiperboli?ki tanges</li>
    <li>Isprobati ne?to druga?ije arhitekture mre?e</li>
    <li>Uporediti rezultate treniranja mre?e kada se pove?a/smanji broj epoha treniranja</li>
</ul>
</div>